In [0]:
from pyspark.sql.functions import *
print("[STEP 0] Imports loaded successfully")

In [0]:
dbutils.widgets.text('source_table_name','')
dbutils.widgets.text('landing_timestamp','')
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

table_name  = dbutils.widgets.get('source_table_name')
landing_timestamp = dbutils.widgets.get('landing_timestamp')
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("[STEP 1] Widget parameters loaded successfully")
print(f"[PARAM] source_table_name = '{table_name}'")
print(f"[PARAM] landing_timestamp = '{landing_timestamp}'")
print(f"[PARAM] merge_keys        = '{merge_keys}'")
print(f"[PARAM] load_type         = '{load_type}'")

In [0]:
source_table = f'vk_adb_caresync_workspace.bronze.{table_name}'
filter_query = f"landing_timestamp = '{landing_timestamp}'"

print(f"[STEP 1.1] Reading bronze table: {source_table}")
print(f"[FILTER] Applying filter: {filter_query}")

df_bronze = spark.read.table(source_table).filter(filter_query)

print(f"[RESULT] Bronze DataFrame loaded successfully")
print(f"[SCHEMA] Columns: {df_bronze.columns}")
print(f"[SCHEMA] Total columns: {len(df_bronze.columns)}")

In [0]:
print(f"[STEP 1.2] Previewing bronze data for table '{table_name}' with landing_timestamp = '{landing_timestamp}'")
record_count_bronze = df_bronze.count()
print(f"[ROW COUNT] {record_count_bronze} records found in bronze after filter")
df_bronze.display()

In [0]:
print("[STEP 1.3] Adding audit columns: 'insert_timestamp' and 'update_timestamp'")
df_bronze_updated = df_bronze.withColumn('insert_timestamp', current_timestamp()).withColumn('update_timestamp', current_timestamp())
print(f"[RESULT] Updated DataFrame columns: {df_bronze_updated.columns}")
print(f"[RESULT] Total columns after update: {len(df_bronze_updated.columns)}")
df_bronze_updated.display()

In [0]:
print("[STEP 2] Preparing load metadata")
target_table = f"vk_adb_caresync_workspace.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print(f"[TARGET TABLE] {target_table}")
print(f"[RAW LOAD TYPE] '{load_type}'")
print(f"[NORMALIZED LOAD TYPE] {normalized_load_type}")
print(f"[MERGE KEYS] {merge_key_list}")

if normalized_load_type == "MERGE":
    print("[MERGE VALIDATION] Validating merge keys against source columns")
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in df_bronze.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")

    print("[MERGE VALIDATION COMPLETE] All merge keys are present in source data")

updated_df = (
    df_bronze.withColumn("insert_timestamp", current_timestamp())
    .withColumn("update_timestamp", current_timestamp())
)

print("[STEP 2 COMPLETE] Added insert_timestamp and update_timestamp columns")
print(f"[UPDATED COLUMNS] {updated_df.columns}")

In [0]:
print(f"[STEP 3] Starting write to target table: {target_table}")

if normalized_load_type == "FULL":
    print("[WRITE MODE] FULL")
    print(f"[WRITE ACTION] Overwriting target table {target_table}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("[STEP 3 COMPLETE] FULL overwrite finished successfully")
elif normalized_load_type == "APPEND":
    print("[WRITE MODE] APPEND")
    print(f"[WRITE ACTION] Appending records into target table {target_table}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("[STEP 3 COMPLETE] APPEND finished successfully")
elif normalized_load_type == "MERGE":
    print("[WRITE MODE] MERGE")
    target_exists = spark.catalog.tableExists(target_table)
    print(f"[TARGET EXISTS] {target_exists}")

    if not target_exists:
        print("[MERGE FALLBACK] Target table does not exist. Creating target table with current dataset.")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("[STEP 3 COMPLETE] Target table created for initial MERGE load")
    else:
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"[TEMP VIEW] Created temp view '{source_view}' for merge source")

        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_key_list]
        )
        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """.strip()

        print("[MERGE SQL]")
        print(merge_sql)

        spark.sql(merge_sql)
        print("[STEP 3 COMPLETE] MERGE finished successfully")
else:
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")

In [0]:
print("[STEP 4] Preparing notebook exit")
record_count = df_bronze_updated.count()
print(f"[FINAL COUNT] Total records processed: {record_count}")
print(f"[EXIT] Exiting notebook with record count: {record_count}")
dbutils.notebook.exit(str(record_count))